<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2005%20-%202026-04-27%20-%20NumPy/clase_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 5 · NumPy — 🧪 Taller

**Fecha:** 27 de abril de 2026 · **Duración estimada:** ~60 min · **Nivel:** Intermedio

> 📖 **La teoría completa vive en el [HTML de la sesión](./index.html).** Este cuaderno es el **taller práctico**: aquí codificas.

## 🎯 Objetivos de aprendizaje

Al terminar este cuaderno serás capaz de:

1. **Construir** arrays n-dimensionales y manipular su forma con `reshape` y `transpose`.
2. **Implementar** operaciones vectorizadas y entender cuándo y cómo el broadcasting actúa.
3. **Diagnosticar** cuellos de botella en código numérico comparando bucles vs. vectorización.
4. **Analizar** estadísticas básicas y filtrados booleanos en datos numéricos.

## ✅ Prerrequisitos

- Haber completado [Día 4](../Dia%2004%20-%202026-04-24%20-%20Planteamiento%20del%20proyecto/) · tipos de datos y listas Python.
- Entender qué es un bucle y cuándo es más lento que una operación vectorizada.

## 🧭 Cómo usar este cuaderno

- Ejecuta cada celda con `Shift + Enter`.
- Las celdas 🟢🟡🔴 son ejercicios (verde → rojo = dificultad creciente).
- Las celdas `# %% [solution]` contienen la solución oculta: solo descomenta si estás atascado.
- Hay `assert` al final de cada ejercicio: si no falla, está correcto.


In [ ]:
# --- Setup del entorno ---------------------------------------------------
# Imports estándar, reproducibilidad y confirmación visual
# ------------------------------------------------------------------------

# Instalación (descomentar solo la primera vez en Colab)
# !pip install --quiet numpy pandas matplotlib seaborn

# Stdlib
import random
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Estilo visual coherente
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"numpy {np.__version__} · pandas {pd.__version__}")


## 🎬 Hook · Normalizar datos con una línea (sin bucles)

> **Pregunta de hoy:** ¿cómo centramos y escalamos 1.000.000 de puntos en 0.3 segundos?

La respuesta: vectorización + broadcasting. Veamos el resultado primero:


In [ ]:
# Datos simulados: 1M de puntos en 3D
np.random.seed(42)
datos_crudos = np.random.randn(1_000_000, 3) * 100 + 50

print(f"Shape original: {datos_crudos.shape}")
print(f"Primeras filas: \n{datos_crudos[:3]}")
print(f"Media por columna: {datos_crudos.mean(axis=0).round(2)}")
print(f"Std por columna: {datos_crudos.std(axis=0).round(2)}")

# ⭐ Normalización con UNA LÍNEA (broadcasting)
datos_normalizados = (datos_crudos - datos_crudos.mean(axis=0)) / datos_crudos.std(axis=0)

print(f"\nDespués de normalizar:")
print(f"Media: {datos_normalizados.mean(axis=0).round(6)}")
print(f"Std: {datos_normalizados.std(axis=0).round(6)}")


⏱️ ~5 min · Demo

> 💡 **Concepto clave:** El broadcasting automático permite restar un vector de una matriz sin necesidad de bucles. NumPy entiende las dimensiones y replica internamente.


---

## 🟢 Ejercicio 1 — Guiado · Crear arrays con formas específicas

⏱️ ~7 min · Patrón: fill-in-the-blank

**Output esperado:** tres arrays: uno (5,), uno (3, 4) y uno (2, 2, 3).


In [ ]:
# 🟢 Completa los huecos con la función correcta

# Vector de 5 ceros
vec = np.___((5,))

# Matriz 3×4 de unos
mat = np.___((3, 4))

# Tensor 2×2×3 con valores aleatorios uniformes [0,1)
tens = np.___((2, 2, 3))

print(f"Vec shape: {vec.shape}, dtype: {vec.dtype}")
print(f"Mat shape: {mat.shape}, dtype: {mat.dtype}")
print(f"Tens shape: {tens.shape}, dtype: {tens.dtype}")


In [ ]:
# Validación automática
assert "vec" in dir(), "La variable vec no existe"
assert vec.shape == (5,), f"vec debe tener shape (5,), tienes {vec.shape}"
assert mat.shape == (3, 4), f"mat debe tener shape (3, 4), tienes {mat.shape}"
assert tens.shape == (2, 2, 3), f"tens debe tener shape (2, 2, 3), tienes {tens.shape}"
print("✅ Ejercicio 1 completado")


In [ ]:
# %% [solution]
# vec = np.zeros((5,))
# mat = np.ones((3, 4))
# tens = np.random.rand(2, 2, 3)


⏸️ **Pausa y reflexión:** ¿por qué NumPy distingue entre shape `(5,)` y `(1, 5)`? ¿Qué diferencia hay?


---

## 🟡 Ejercicio 2 — Aplicado · Filtrado y indexación booleana

⏱️ ~10 min · Contexto: vendedor analiza precios.

**Dados:** precios y cantidades de 100 productos.  
**Pide:** encontrar productos caros (> percentil 75) con bajo stock (< mediana).


In [ ]:
np.random.seed(42)
precios = np.random.uniform(10, 500, size=100)  # 100 precios entre 10 y 500
stock = np.random.randint(1, 100, size=100)     # 100 cantidades entre 1 y 100

p75 = np._____(precios, 75)  # percentil 75 de precios
stock_mediana = np.median(stock)

# 🟡 Indexación booleana: productos CAROS Y con BAJO stock
mask = (precios > p75) & (stock < stock_mediana)
productos_objetivo = precios[mask]

print(f"Percentil 75 de precios: ${p75:.2f}")
print(f"Mediana de stock: {stock_mediana:.0f} unidades")
print(f"Productos caros + bajo stock: {len(productos_objetivo)}")
print(f"Rango de precios objetivo: ${productos_objetivo.min():.2f} — ${productos_objetivo.max():.2f}")


In [ ]:
# Validación
assert "productos_objetivo" in dir(), "Variable no definida"
assert len(productos_objetivo) > 0, "Deberías encontrar al menos 1 producto"
assert (productos_objetivo > p75).all(), "Todos deben ser > percentil 75"
print("✅ Ejercicio 2 completado")


In [ ]:
# %% [solution]
# p75 = np.percentile(precios, 75)


⏸️ **Pausa:** ¿cuántos productos encontraste? Si llamamos "estrategia" a enfocarse en productos de alto valor con bajo inventario, ¿qué data podrías agregar para mejorar la decisión?


---

## 🔴 Ejercicio 3 — Desafío · Broadcasting + agregaciones

⏱️ ~15 min · Open-ended: sin skeleton.

**Contexto:** tienes una matriz de **distancias euclídeas** (10×10) entre 10 puntos. Quieres identificar si hay **clusters**.

**Pasos esperados:**
1. Crea una matriz de distancias simulada (gaussiana, media=5, std=2).
2. Para cada fila, calcula la distancia media a todos sus vecinos.
3. Identifica los puntos con distancia media **< percentil 25** (candidatos a "centros de cluster").
4. Reporta sus índices y su distancia media.


In [ ]:
np.random.seed(42)
# Tu código aquí
# Crea matrix_distancias (10, 10) con np.random.normal(5, 2, (10, 10))
# Calcula mean_por_fila con axis=?
# Encuentra percentil_25
# Filtra puntos candidatos usando indexación booleana

print("Matriz shape:", matrix_distancias.shape)
print("Distancia media por fila:", mean_por_fila)
print("Puntos candidatos a centros:", puntos_candidatos)


In [ ]:
# Criterios de aceptación
assert "matrix_distancias" in dir() and matrix_distancias.shape == (10, 10)
assert "mean_por_fila" in dir() and mean_por_fila.shape == (10,)
assert "puntos_candidatos" in dir() and len(puntos_candidatos) > 0
print("✅ Ejercicio 3 completado")


In [ ]:
# %% [solution]
# np.random.seed(42)
# matrix_distancias = np.random.normal(loc=5, scale=2, size=(10, 10))
# mean_por_fila = matrix_distancias.mean(axis=1)
# percentil_25 = np.percentile(mean_por_fila, 25)
# puntos_candidatos = np.where(mean_por_fila < percentil_25)[0]


---

## ⚡ Benchmark · Bucles vs. Vectorización

⏱️ ~5 min · Visualizar la diferencia de velocidad.


In [ ]:
import time

def suma_bucle(arr):
    """Suma los elementos con un bucle Python."""
    total = 0.0
    for v in arr:
        total += v
    return total

# Preparar datos
np.random.seed(42)
N = 100_000
x = np.random.randn(N)

# Benchmark suma
t0 = time.time()
suma_python = suma_bucle(x)
t_python = time.time() - t0

t0 = time.time()
suma_numpy = x.sum()
t_numpy = time.time() - t0

print(f"Suma (N={N}):")
print(f"  Python:  {t_python*1000:.2f} ms")
print(f"  NumPy:   {t_numpy*1000:.3f} ms")
print(f"  Speedup: {t_python/t_numpy:.0f}x")


> 📌 **Insight:** este factor 100—1000× es la razón por la que NumPy existe.


---

## 🧭 Checkpoint · Auto-evaluación

1. **Broadcasting:** mecanismo que permite sumar arrays de shapes distintos ← ✓
2. **axis=0** agrega filas, **axis=1** agrega columnas
3. **Vectorización** es 100—1000× más rápido que bucles Python

📊 Has completado **5 de 6 secciones** · ¡Vamos!


In [ ]:
# Validación automática
assert "vec" in dir() and vec.shape == (5,), "Reinicia y completa ejercicio 1"
print("Checkpoint ✓ — puedes continuar")


---

## 🧠 Mental model de la sesión

```
Datos → Array NumPy → Operaciones vectorizadas
             ↓
       Broadcasting automático
             ↓
       Agregaciones (sum, mean, std)
             ↓
       Filtrado booleano (sin bucles)
             ↓
       Resultado (100—1000× más rápido)
```


## 📌 Resumen · Lo que aprendiste hoy

- Arrays NumPy son **homogéneos, contiguos y rápidos** vs. listas Python.
- **Broadcasting** permite operaciones entre shapes distintos sin bucles.
- **Agregaciones** con `axis=0` (filas) y `axis=1` (columnas).
- **Vectorización:** 100—1000× más rápido + código más limpio.
- **Indexación booleana:** `arr[arr > 5]` es tu amiga.


## 🚀 Próximos pasos (D+1)

Mañana veremos **Pandas I** — cargar datos reales, explorarlos y hacer selecciones. NumPy seguirá viviendo debajo, pero raramente lo tocarás directamente.

**Para preparar:**
- Relee [la teoría de broadcasting](./index.html) si te quedó difuso.
- Practica creando arrays raros: `(1, 1, 5)`, `(3, 1)`, etc.


## 📚 Para profundizar

- 📖 [NumPy Docs Oficiales — Absolute Beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)
- 🎥 [NumPy Broadcasting Visual](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- 📝 [Ten NumPy Tips for Data Science](https://towardsdatascience.com/ten-numpy-tips-for-data-science-b29ef6c5c1f5)


## ✍️ Reflexión final

> En tus propias palabras, explícale a un compañero **por qué NumPy es más rápido que Python puro** en máximo 3 oraciones.


---

## 🧑‍🤝‍🧑 Trabajo en grupo — avance del proyecto

**Preparación para Pandas I:**

- Descarguen el dataset de su caso (CSV, Excel o API).
- Reporten:
  - Shape (filas × columnas)
  - Tipos (`dtypes`)
  - % nulos por columna
  - Primeras 5 filas
  - 1 observación sorprendente

**Entregable:** `proyecto_dataset_inicial.ipynb` en su repo.
